In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom

from lungmask import mask as lungmask
from scipy.ndimage import binary_dilation, label
from collections import Counter

def find_best_lung_slice(lung_mask):
    best_idx = 0
    best_score = 0

    for i in range(lung_mask.shape[0]):
        mask = lung_mask[i]

        # must have some lung
        lung_pixels = mask.sum()
        if lung_pixels == 0:
            continue

        # split rough left/right presence
        mid = mask.shape[1] // 2
        left = mask[:, :mid].sum()
        right = mask[:, mid:].sum()

        # score = total lung coverage + balance between sides
        score = lung_pixels + min(left, right)

        if score > best_score:
            best_score = score
            best_idx = i

    return best_idx

# =========================================================
# OUTPUT
# =========================================================
os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/data", exist_ok=True)

# =========================================================
# 1. DICOM LOADER
# =========================================================
def get_dicom_files(root):
    files = []
    for r, _, fns in os.walk(root):
        for f in fns:
            if f.lower().endswith(".dcm"):
                files.append(os.path.join(r, f))
    return sorted(files)

# =========================================================
# 2. CT LOADER (HU CORRECT)
# =========================================================
def load_ct(files):
    slices, shapes = [], []

    for f in files:
        try:
            ds = pydicom.dcmread(f)
            img = ds.pixel_array.astype(np.float32)

            slope = getattr(ds, "RescaleSlope", 1.0)
            intercept = getattr(ds, "RescaleIntercept", 0.0)

            img = img * slope + intercept

            slices.append(img)
            shapes.append(img.shape)

        except:
            continue

    if len(slices) == 0:
        raise RuntimeError("❌ No CT loaded")

    common_shape = Counter(shapes).most_common(1)[0][0]
    slices = [s for s in slices if s.shape == common_shape]

    ct = np.stack(slices, axis=0)

    print("✅ CT loaded:", ct.shape)
    print("CT min/max:", ct.min(), ct.max())

    return np.clip(ct, -1000, 4000)

# =========================================================
# 3. PET LOADER
# =========================================================
def load_pet(files):
    slices, shapes = [], []

    for f in files:
        try:
            ds = pydicom.dcmread(f)
            img = ds.pixel_array.astype(np.float32)

            slices.append(img)
            shapes.append(img.shape)

        except:
            continue

    if len(slices) == 0:
        raise RuntimeError("❌ No PET loaded")

    common_shape = Counter(shapes).most_common(1)[0][0]
    slices = [s for s in slices if s.shape == common_shape]

    pet = np.stack(slices, axis=0)

    print("✅ PET loaded:", pet.shape)
    print("PET min/max:", pet.min(), pet.max())

    return pet

# =========================================================
# 4. LUNG SEGMENTATION (ROBUST + FALLBACK)
# =========================================================
def segment_lungs(ct):
    mask = lungmask.apply(ct)
    mask = (mask > 0).astype(np.uint8)

    coverage = mask.mean()
    print("Lung coverage:", coverage)

    # fallback if lungmask fails
    if coverage < 0.01:
        print("⚠️ fallback thresholding")
        mask = (ct < np.percentile(ct, 45)).astype(np.uint8)

    if mask.mean() < 0.01:
        print("⚠️ dilation fallback")
        mask = binary_dilation(mask, iterations=2)

    return mask

# =========================================================
# 5. KEEP LARGEST COMPONENT
# =========================================================
def keep_largest(mask):
    labeled, n = label(mask)

    if n == 0:
        return mask

    sizes = [(labeled == i).sum() for i in range(1, n + 1)]
    largest = np.argmax(sizes) + 1

    return (labeled == largest).astype(np.uint8)

# =========================================================
# 6. PET DISPLAY NORMALIZATION (IMPORTANT FIX)
# =========================================================
def normalize_pet(img):
    vmin = np.percentile(img, 1)
    vmax = np.percentile(img, 99)

    img = np.clip(img, vmin, vmax)
    return (img - vmin) / (vmax - vmin + 1e-8)

# =========================================================
# 7. UPTAKE DETECTION (SUV-LIKE ROBUST)
# =========================================================
def detect_uptake(pet, lung_mask):
    vals = pet[lung_mask > 0]

    if len(vals) < 50:
        raise RuntimeError("❌ insufficient lung voxels")

    mean = np.mean(vals)
    std = np.std(vals)

    threshold = np.percentile(vals, 95) if std < 1e-6 else mean + 2 * std

    uptake = (pet > threshold) & (lung_mask == 1)

    return uptake, threshold

# =========================================================
# 8. METRICS
# =========================================================
def compute_metrics(pet, mask):
    vals = pet[mask > 0]

    if len(vals) == 0:
        return {"volume": 0, "max": 0, "mean": 0}

    return {
        "volume": int(mask.sum()),
        "max": float(vals.max()),
        "mean": float(vals.mean())
    }

# =========================================================
# 9. FULL VISUALIZATION (FIXED)
# =========================================================
def show_full_view(ct, pet, lung_mask, uptake_mask, idx):

    fig, ax = plt.subplots(1, 4, figsize=(20, 5))

    # CT
    ct_img = np.clip(ct[idx], -1000, 400)
    ax[0].imshow(ct_img, cmap="gray")
    ax[0].set_title("CT")
    ax[0].axis("off")

    # Lungs
    lung_img = np.zeros_like(ct_img)
    lung_img[lung_mask[idx] == 1] = ct_img[lung_mask[idx] == 1]

    ax[1].imshow(lung_img, cmap="gray")
    ax[1].set_title("Lungs")
    ax[1].axis("off")

    # PET normalized
    pet_img = normalize_pet(pet[idx])
    ax[2].imshow(pet_img, cmap="gray")
    ax[2].set_title("PET")
    ax[2].axis("off")

    # Tumor heatmap
    ax[3].imshow(pet_img, cmap="gray")

    ax[3].imshow(
        np.ma.masked_where(~uptake_mask[idx], uptake_mask[idx]),
        cmap="coolwarm",
        alpha=0.6
    )

    ax[3].set_title("Tumor Heatmap")
    ax[3].axis("off")

    plt.tight_layout()
    plt.show()

# =========================================================
# 10. MAIN PIPELINE
# =========================================================

BASE = "/Users/Marcin_1/Desktop/PET/NSCLC/NSCLC Radiogenomics/R01-018"

files = get_dicom_files(BASE)

ct_files = [f for f in files if "CT" in f.upper()]
pet_files = [f for f in files if "PET" in f.upper()]

print("CT files:", len(ct_files))
print("PET files:", len(pet_files))

# LOAD
ct = load_ct(ct_files)
pet = load_pet(pet_files)

# LUNG SEGMENTATION
lung_mask = segment_lungs(ct)
lung_mask = keep_largest(lung_mask)

# UPTAKE DETECTION
uptake_mask, thr = detect_uptake(pet, lung_mask)

print("\nSUV threshold:", thr)
print("Tumor voxels:", uptake_mask.sum())
print("Tumor ratio:", uptake_mask.mean())

# METRICS
result = compute_metrics(pet, uptake_mask)

df = pd.DataFrame([{
    "patient": "R01-018",
    "threshold": thr,
    "volume": result["volume"],
    "max": result["max"],
    "mean": result["mean"]
}])

df.to_csv("outputs/data/R01-018_metrics.csv", index=False)

print("📊 Saved metrics")

# VISUAL CHECK (SAFE INDEX)
idx = find_best_lung_slice(lung_mask)
print("Best lung slice:", idx)

show_full_view(ct, pet, lung_mask, uptake_mask, idx)